# Amodal Fine-tuning -- Full Pipeline (Mask + Appearance)

Chay toan bo tren 1 notebook Kaggle, tu dau den cuoi.

**Truoc khi chay:** Settings -> Accelerator -> GPU T4 x1.

Cau truc notebook:
1. Setup (clone repo, cai dependencies)
2. Nhanh Mask -- du lieu synthetic + train U-Net + danh gia IoU
3. Nhanh Appearance -- dataset DreamBooth + train LoRA + danh gia CLIP score
4. Tong hop ket qua + luu checkpoint de tai ve

## 1. Setup

In [ ]:
!git clone https://github.com/YouttyLe-DSAI/LAOC2WAM-Learning-Amodal-Object-Completion-With-World-Action-Model.git repo
%cd repo

In [ ]:
!pip install -q -r requirements.txt
!pip install -q datasets
# Fix xung dot version peft/torchao tren moi truong Kaggle
!pip uninstall -y -q torchao

In [ ]:
import os
os.environ["PYTHONPATH"] = "."
import torch
print("GPU available:", torch.cuda.is_available())

## 2. Nhanh Mask -- du doan amodal mask

In [ ]:
# Tao du lieu synthetic occlusion (tu dong tai anh mau + tu ve occluder gia)
!PYTHONPATH=. python scripts/01b_prepare_synthetic.py --images data/raw --out data/cocoa \
    --download_samples --n_occluders_per_image 8

In [ ]:
# Train U-Net du doan amodal mask (nhe, chay nhanh ngay ca tren GPU T4)
!PYTHONPATH=. python scripts/04a_train_mask_model.py --config configs/config.yaml --epochs 100

In [ ]:
# Danh gia IoU: so sanh voi baseline (chi dung visible mask, khong fine-tune)
!PYTHONPATH=. python scripts/06_evaluate_mask.py --config configs/config.yaml \
    --checkpoint outputs/mask_model/best.pt

## 3. Nhanh Appearance -- LoRA fine-tune (DreamBooth style)

In [ ]:
# Tai dataset DreamBooth chinh thuc (public, khong can quyen)
!git clone https://github.com/google/dreambooth.git third_party/dreambooth-dataset
!ls third_party/dreambooth-dataset/dataset/

In [ ]:
# Doi SUBJECT o day de thu subject khac, xem danh sach o cell tren
SUBJECT = "backpack"
!PYTHONPATH=. python scripts/02b_prepare_dreambooth_data.py \
    --dreambooth_dir third_party/dreambooth-dataset \
    --subject {SUBJECT} \
    --out data/train_ready

In [ ]:
# Train LoRA -- mat khoang 2-3 phut tren GPU T4 voi 100 steps mac dinh
!PYTHONPATH=. python scripts/04_train_lora.py --config configs/config.yaml

In [ ]:
!ls outputs/lora/final
!zip -rq lora_{SUBJECT}.zip outputs/lora/final
print(f"Da nen checkpoint: lora_{SUBJECT}.zip -- nho tai ve truoc khi het session")

### 3.1. Sinh anh demo de kiem tra truc quan

In [ ]:
from diffusers import StableDiffusionPipeline
from peft import PeftModel

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
).to("cuda")
pipe.unet = PeftModel.from_pretrained(pipe.unet, "outputs/lora/final")

demo_prompt = f"a photo of sks {SUBJECT} on a beach"
demo_image = pipe(demo_prompt, num_inference_steps=30).images[0]
demo_image.save("demo_output.png")
demo_image

### 3.2. Danh gia CLIP similarity (subject fidelity)

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import glob

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

ref_paths = sorted(glob.glob("data/train_ready/instance_images/*.jpg"))
sims = []
for ref_path in ref_paths:
    ref_image = Image.open(ref_path).convert("RGB")
    inputs = clip_processor(images=[demo_image, ref_image], return_tensors="pt").to("cuda")
    with torch.no_grad():
        vision_out = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        embeds = clip_model.visual_projection(vision_out.pooler_output)
    embeds = embeds / embeds.norm(dim=-1, keepdim=True)
    sims.append((embeds[0] @ embeds[1]).item())

print(f"CLIP similarity trung binh voi {len(ref_paths)} anh goc: {sum(sims)/len(sims):.4f}")
print(f"Chi tiet tung anh: {[round(s,4) for s in sims]}")

## 4. Tong hop ket qua

In [ ]:
print("===== TONG HOP KET QUA =====")
print("\n[Nhanh Mask]")
print("Xem output cell danh gia IoU o Phan 2 (IoU sau fine-tune vs baseline)")
print("\n[Nhanh Appearance]")
print(f"Subject: sks {SUBJECT}")
print(f"CLIP similarity trung binh: {sum(sims)/len(sims):.4f}")
print("Checkpoint LoRA: outputs/lora/final (da nen thanh lora_{}.zip)".format(SUBJECT))
print("\n==> Nho tai ve: lora_{}.zip va demo_output.png truoc khi dong notebook".format(SUBJECT))